In [ ]:
!pip install gspread oauth2client geopy pandas numpy -q

import gspread
from google.colab import auth
from google.auth import default
import pandas as pd
import numpy as np
from datetime import datetime

print("All libraries installed successfully!")

class DataLoader:
    """Load and prepare all data from Google Sheets"""

    def __init__(self, spreadsheet_name):
        self.spreadsheet_name = spreadsheet_name
        self.authenticate()

    def authenticate(self):
        auth.authenticate_user()
        creds, _ = default()
        self.gc = gspread.authorize(creds)
        print("\n Authentication successful\n")

    def load_all_data(self):
        print("=" * 60)
        print("LOADING DATA FROM GOOGLE SHEETS")
        print("=" * 60)

        spreadsheet = self.gc.open(self.spreadsheet_name)
        data = {}

        depot_ws = spreadsheet.worksheet("Depot")
        data['depot'] = pd.DataFrame(depot_ws.get_all_records())
        print(f"\n1. Loaded: {len(data['depot'])} depot")

        vehicles_ws = spreadsheet.worksheet("Vehicles")
        data['vehicles'] = pd.DataFrame(vehicles_ws.get_all_records())
        print(f"\n2. Loaded: {len(data['vehicles'])} vehicles")

        customers_ws = spreadsheet.worksheet("Customers")
        data['customers'] = pd.DataFrame(customers_ws.get_all_records())
        print(f"\n3. Loaded: {len(data['customers'])} customers")

        lockers_ws = spreadsheet.worksheet("Lockers")
        data['lockers'] = pd.DataFrame(lockers_ws.get_all_records())
        print(f"\n4. Loaded: {len(data['lockers'])} lockers")

        distance_ws = spreadsheet.worksheet("Distance_Matrix_500")
        data['distances'] = pd.DataFrame(distance_ws.get_all_records())
        print(f"\n5. Loaded: {len(data['distances'])} distances")

        print("\n ALL DATA LOADED SUCCESSFULLY")

        return data

    def preprocess_data(self, data):
        print("\n" + "=" * 60)
        print("PREPROCESSING DATA")
        print("=" * 60)

        processed = {}

        depot = data['depot'].iloc[0]
        processed['depot'] = {
            'id': depot['Depot_ID'],
            'lat': depot['Latitude'],
            'lon': depot['Longitude'],
            'district': depot.get('District', 'Unknown')
        }
        print(f"\n Depot: {processed['depot']['id']}")

        df_customers = data['customers'].copy()
        df_customers['is_locker_customer'] = df_customers['Locker_Allowed_Binary'] == 1

        processed['customers_home'] = df_customers[~df_customers['is_locker_customer']].copy()
        processed['customers_locker'] = df_customers[df_customers['is_locker_customer']].copy()

        print(f"\n Initial customer split:")
        print(f"  - Home delivery: {len(processed['customers_home'])}")
        print(f"  - Locker preference: {len(processed['customers_locker'])}")

        processed['lockers'] = data['lockers'].copy()
        print(f"\n Lockers: {len(processed['lockers'])}")

        processed['vehicles'] = data['vehicles'].copy()
        processed['vehicles_by_type'] = {
            'Bicycle': data['vehicles'][data['vehicles']['Vehicle_Type'] == 'Bicycle'],
            'Electric': data['vehicles'][data['vehicles']['Vehicle_Type'] == 'Electric'],
            'Diesel': data['vehicles'][data['vehicles']['Vehicle_Type'] == 'Diesel']
        }

        print(f"\n Vehicles: {len(data['vehicles'])}")

        df_dist = data['distances'].copy()
        processed['distance_dict'] = {}
        for _, row in df_dist.iterrows():
            key = (row['From_ID'], row['To_ID'])
            processed['distance_dict'][key] = row['Distance_km']

        print(f"\n Distance Matrix: {len(df_dist)} distances")

        processed['zones'] = {
            'Bicycle': {'min': 0, 'max': 2.5},
            'Electric': {'min': 2.5, 'max': 7},
            'Diesel': {'min': 7, 'max': float('inf')}
        }

        print("\n Zone definitions:")
        for zone_type, bounds in processed['zones'].items():
            print(f"   - {zone_type}: {bounds['min']}-{bounds['max']} km")

        processed['params'] = {
            'service_time_home_base': 3,
            'service_time_home_per': 1,
            'service_time_locker_base': 1,
            'service_time_locker_per': 0.50,

            'max_shift_hours': 8,
            'max_shift_minutes': 480,
            'max_walking_distance': 2.0
        }

        print("\n Parameters:")
        print(f"   - Service time (home base): {processed['params']['service_time_home_base']} min")
        print(f"   - Service time (home per parcel): {processed['params']['service_time_home_per']} min")
        print(f"   - Service time (locker base): {processed['params']['service_time_locker_base']} min")
        print(f"   - Service time (locker per parcel): {processed['params']['service_time_locker_per']} min")
        print(f"   - Max walking distance: {processed['params']['max_walking_distance']} km")
        print(f"   - Max shift: {processed['params']['max_shift_hours']} hours")

        print("\n" + "=" * 60)
        print("FEASIBLE LOCKER ASSIGNMENTS")
        print("=" * 60)

        max_walk = processed['params']['max_walking_distance']
        feasible_assignments = {}
        total_feasible = 0
        customers_to_reclassify = []

        for _, customer in processed['customers_locker'].iterrows():
            customer_id = customer['Customer_ID']
            feasible_lockers = []

            for _, locker in processed['lockers'].iterrows():
                locker_id = locker['Locker_ID']

                try:
                    walking_distance = self.get_distance(
                        customer_id,
                        locker_id,
                        processed['distance_dict']
                    )

                    if walking_distance <= max_walk:
                        feasible_lockers.append({
                            'locker_id': locker_id,
                            'walking_distance': walking_distance
                        })
                        total_feasible += 1

                except ValueError:
                    continue

            if len(feasible_lockers) == 0:
                customers_to_reclassify.append(customer_id)
            else:
                feasible_assignments[customer_id] = feasible_lockers

        if len(customers_to_reclassify) > 0:
            print(f"\n  Reclassifying {len(customers_to_reclassify)} customers to HOME delivery:")
            for cust_id in customers_to_reclassify:
                print(f"   - {cust_id} (no lockers within {max_walk} km)")

                cust_row = processed['customers_locker'][
                    processed['customers_locker']['Customer_ID'] == cust_id
                ]

                if not cust_row.empty:
                    processed['customers_home'] = pd.concat([
                        processed['customers_home'],
                        cust_row
                    ], ignore_index=True)

                    processed['customers_locker'] = processed['customers_locker'][
                        processed['customers_locker']['Customer_ID'] != cust_id
                    ]

        processed['feasible_assignments'] = feasible_assignments

        customers_with_lockers = len(processed['customers_locker'])

        print(f"\n Feasible assignments:")
        print(f"   - Locker customers (can use lockers): {customers_with_lockers}")
        print(f"   - Reclassified to home: {len(customers_to_reclassify)}")

        print("\n" + "=" * 60)
        print("ASSIGNING CUSTOMERS TO LOCKERS (Capacity-Aware Assignment)")
        print("=" * 60)

        locker_capacity = dict(zip(
            processed['lockers']['Locker_ID'],
            processed['lockers']['Locker_Capacity']
        ))
        locker_usage = {l: 0 for l in locker_capacity}

        locker_assignments = {}
        fallback_to_home = []

        customer_priority = []
        for cust_id, feasible in feasible_assignments.items():
            num_available = len([l for l in feasible
                                if locker_usage[l['locker_id']] < locker_capacity[l['locker_id']]])
            customer_priority.append((cust_id, num_available, feasible))

        customer_priority.sort(key=lambda x: x[1])

        for cust_id, _, feasible in customer_priority:
            sorted_feasible = sorted(feasible, key=lambda x: x['walking_distance'])

            assigned = False
            for locker_option in sorted_feasible:
                locker_id = locker_option['locker_id']

                if locker_usage[locker_id] < locker_capacity[locker_id]:
                    locker_assignments[cust_id] = locker_id
                    locker_usage[locker_id] += 1
                    assigned = True
                    break

            if not assigned:
                fallback_to_home.append(cust_id)

        print(f"\n Locker assignments: {len(locker_assignments)}")
        print(f" Fallback to home (capacity): {len(fallback_to_home)}")

        if len(fallback_to_home) > 0:
            print(f"\n  Reclassifying {len(fallback_to_home)} more customers (capacity):")
            for cust_id in fallback_to_home:
                print(f"   - {cust_id} (all feasible lockers full)")

                cust_row = processed['customers_locker'][
                    processed['customers_locker']['Customer_ID'] == cust_id
                ]

                if not cust_row.empty:
                    processed['customers_home'] = pd.concat([
                        processed['customers_home'],
                        cust_row
                    ], ignore_index=True)

                    processed['customers_locker'] = processed['customers_locker'][
                        processed['customers_locker']['Customer_ID'] != cust_id
                    ]

        processed['locker_assignments'] = locker_assignments

        used_lockers = [l for l, u in locker_usage.items() if u > 0]
        print(f"\n Lockers used: {len(used_lockers)}/{len(processed['lockers'])}")
        for locker_id in sorted(used_lockers):
            usage = locker_usage[locker_id]
            capacity = locker_capacity[locker_id]
            print(f"   - {locker_id}: {usage}/{capacity} ({usage/capacity*100:.0f}%)")

        print(f"\n FINAL customer distribution:")
        print(f"   - Home delivery: {len(processed['customers_home'])}")
        print(f"   - Locker delivery: {len(processed['customers_locker'])}")
        print(f"   - Total: {len(processed['customers_home']) + len(processed['customers_locker'])}")

        print("\n" + "=" * 60)
        print(" DATA PREPROCESSING COMPLETE")
        print("=" * 60)

        return processed

    def get_distance(self, from_id, to_id, distance_dict):
        key = (from_id, to_id)
        if key in distance_dict:
            return distance_dict[key]
        reverse_key = (to_id, from_id)
        if reverse_key in distance_dict:
            return distance_dict[reverse_key]
        raise ValueError(f"Distance not found for {from_id} -> {to_id}")

    def validate_data(self, processed):

        errors = []

        for cust_id in processed['customers_locker']['Customer_ID']:
            if cust_id in processed['feasible_assignments']:
                if len(processed['feasible_assignments'][cust_id]) == 0:
                    errors.append(f"Customer {cust_id} in locker list but has no feasible lockers!")
            else:
                errors.append(f"Customer {cust_id} missing from feasible_assignments!")

        for cust_id in processed['customers_locker']['Customer_ID']:
            if cust_id not in processed['locker_assignments']:
                errors.append(f"Customer {cust_id} not assigned to any locker!")

        if errors:
            print("\n VALIDATION FAILED:")
            for error in errors:
                print(f"   - {error}")
            raise ValueError("Data validation failed")

        print("\n All validation checks passed")

        return True

def main():
    SPREADSHEET_NAME = "R1 MD VRP 500 Mixed Fleet"  # Input File

    loader = DataLoader(SPREADSHEET_NAME)
    raw_data = loader.load_all_data()
    processed_data = loader.preprocess_data(raw_data)
    loader.validate_data(processed_data)

    print("\n DATA LOADING COMPLETE!\n")
    return processed_data

if __name__ == "__main__":
    processed_data = main()